# C4 lab: dense retrieval that returns scores

Lesson: [`concepts/C4_dense_retrieval.md`](../concepts/C4_dense_retrieval.md), Part H.1. Concept card: [02 C4](../02_RAG_MASTER_LEARNING_PLAN.md#c4). Prerequisites: your P5 lab (`top_k` with a tie rule) and your P6 lab (Hit@k).

**How this lab works (blank-file first).**
- Each task has a **task cell**, an **empty code cell** for you, and a **CHECK cell** (`# CHECK`, do not edit). Run the CHECK after your code. If it fails, read the message and fix your code.
- This notebook contains **no solution code**. You may reuse **your own** `top_k` from the P5 lab. The reference script `sources/verification/C4/c4_measure.py` is a solution: open it only after an honest attempt and a hint from the tutor (CR-04).
- **Predict first** where asked. The CHECK cells refuse to run without your prediction.
- CHECK cells compare **fingerprints** (`_fp`), so the expected answers stay hidden until you have predicted.
- **No API calls by default.** Task 8 (Cohere) is optional and switched off (`RUN_COHERE = False`). Never print a key.
- Kernel: the project uv environment (`rag-learning`). Run the cells top to bottom.

## Setup (given)

Imports, paths and the helper the CHECK cells use: `_fp` (a fingerprint).

In [ ]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # never download; the models are in the local HF cache
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
import hashlib, json, re, sys, types, dataclasses, tempfile
from pathlib import Path
import numpy as np
import pymupdf

WS = Path(r"C:\Yash\New-Work\Book RAG\rag_learning")
PDF = r"C:\Yash\New-Work\Book RAG\upstream\RAG_Techniques\data\Understanding_Climate_Change.pdf"
MINILM = "sentence-transformers/all-MiniLM-L6-v2"
K = 4                                                  # v0 k (06 §3.1)

def _fp(x):
    """Fingerprint of a value. The CHECK cells compare fingerprints, not visible answers."""
    return hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()[:16]

print("numpy", np.__version__, "| pymupdf", pymupdf.VersionBind)

## The v0 index (given, see C3)

This cell builds the index that **C3** teaches you to build, so this lab can run on its own:
- PyMuPDF `page.get_text()` per page, pages joined with `"\n"`, snapshot id by the 06 §5.3 recipe;
- the 07 H.1/H.3 recursive chunker, `per_page`, 1000/200 characters (executed from 07, as the acceptance suite does): **97** chunks, the same spans as your C2 lab;
- chunk ids `climate:ea07572562ac:1ca9c5a3b548:00000` … `00096`: the 06 §2.3 format, with `params_hash` over 06 §2.4's `chunking:` block (the course-wide ids, as in C2 and C3);
- `chunks`: one dict per chunk with the 06 §2.3 fields (`chunk_id`, `seq`, `char_start`, `char_end`, `page_segments`, `text`, ...);
- `V`: MiniLM vectors, float32, unit rows, **row i = chunks[i]**; `manifest`: what built the index;
- `embedder`: an object with `embed_query(text)` and `embed_documents(texts)` (the 06 §2.3 `Embedder`).

**If you already built the C3 index files**, set `INDEX_DIR` to that folder (it must hold `vectors.npy`, `chunks.jsonl`, `manifest.json` for this exact config; the course layout is `index/<snapshot_id>/<params_hash>/<embed_hash>/`) and the cell loads them instead. Row i of `vectors.npy` is line i of `chunks.jsonl`; `seq` restarts per document, so never assume `seq == i`.

In [ ]:
INDEX_DIR = None        # e.g. Path(r"...\index\ea07572562ac\1ca9c5a3b548\c398eeba043f") once your C3 index exists

def _section(md, prefix):
    """Text under the first heading starting with `prefix`, up to the next heading of the same or higher level."""
    lines, fence, start, level = md.splitlines(), False, None, None
    for i, ln in enumerate(lines):
        if ln.lstrip().startswith("```"):
            fence = not fence; continue
        m = None if fence else re.match(r"^(#+)\s+(.*)$", ln)
        if m and start is None and m.group(2).startswith(prefix):
            start, level = i, len(m.group(1))
        elif m and start is not None and len(m.group(1)) <= level:
            return "\n".join(lines[start:i])
    return "\n".join(lines[start:])

def _blocks(md, lang):
    return re.findall(r"^```" + lang + r"[ \t]*\n(.*?)^```", md, flags=re.S | re.M)

pages = [p.get_text() for p in pymupdf.open(PDF)]
doc_text = "\n".join(pages)
starts = [sum(len(x) + 1 for x in pages[:i]) for i in range(len(pages))]
_sha = lambda s: hashlib.sha256(s.encode("utf-8")).hexdigest()
_parts = ["snapshot-v1", f"parser=pymupdf@{pymupdf.VersionBind}", "options=" + json.dumps({}, sort_keys=True),
          "cleaning=none", "page_join=\\n", f"climate:{_sha(doc_text)}"]          # 06 §5.3 recipe
SNAP = _sha("|".join(_parts))[:12]
assert SNAP == "ea07572562ac", f"snapshot {SNAP}: the extracted text differs from Corpus D (PyMuPDF must be 1.26.7)"

md07 = (WS / "07_SAMPLE_LESSON.md").read_text(encoding="utf-8")
for name in ("ragkit", "ragkit._core", "ragkit.chunking"):
    sys.modules[name] = types.ModuleType(name)
sys.modules["ragkit"].__path__ = []
exec(_blocks(_section(md07, "H.1 Minimal implementation"), "python")[0], sys.modules["ragkit._core"].__dict__)
exec(_blocks(_section(md07, "H.3 Reusable design"), "python")[0], sys.modules["ragkit.chunking"].__dict__)
_ch = sys.modules["ragkit.chunking"]
CFG = _ch.ChunkerConfig(name="recursive", size=1000, overlap=200, scope="per_page")   # the spans (= C2 lab spans)
# chunk ids: 06 §2.3 format; params_hash over 06 §2.4's `chunking:` block (the same ids as the C2 lab and C3)
CHUNKING = {"name": "recursive_char", "unit": "chars", "chunk_size": 1000, "chunk_overlap": 200,
            "separators": ["\n\n", "\n", " ", ""], "scope": "per_page"}
PARAMS_HASH = hashlib.sha256(json.dumps(CHUNKING, sort_keys=True).encode("utf-8")).hexdigest()[:12]
assert PARAMS_HASH == "1ca9c5a3b548", PARAMS_HASH

def _record(c):
    """One 06 §2.3 Chunk record (the chunks.jsonl line C2/C3 write)."""
    return {"doc_id": "climate", "snapshot_id": SNAP, "chunk_id": f"climate:{SNAP}:{PARAMS_HASH}:{c.seq:05d}",
            "seq": c.seq, "char_start": c.char_start, "char_end": c.char_end,
            "page_segments": [{"page": g.page, "char_start": g.char_start, "char_end": g.char_end} for g in c.page_segments],
            "text": c.text, "text_sha256": c.text_sha256, "chunker": CHUNKING["name"], "params_hash": PARAMS_HASH, "meta": {}}

from sentence_transformers import SentenceTransformer

class MiniLMEmbedder:
    """06 §2.3 Embedder for the local fallback model. MiniLM is symmetric: queries and documents use the same call."""
    def __init__(self, name=MINILM):
        self.model_id, self._m = name, SentenceTransformer(name, device="cpu")
        self.dim = self._m.get_sentence_embedding_dimension()
    def embed_documents(self, texts):
        return self._m.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    def embed_query(self, text):
        return self._m.encode([text], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)[0]

embedder = MiniLMEmbedder()
if INDEX_DIR is None:
    chunks = [_record(c) for c in _ch.chunk_document("climate", SNAP, pages, CFG)]
    V = embedder.embed_documents([c["text"] for c in chunks])
    _emb = {"provider": "sentence_transformers", "model_id": MINILM, "revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
            "input_type": {"documents": "none", "queries": "none"}, "normalize": True, "preproc_version": "none"}
    manifest = {"snapshot_id": SNAP, "chunking": CHUNKING, "params_hash": PARAMS_HASH, "n_chunks": len(chunks),
                "embedding": {**_emb, "dim": 384},
                "embed_hash": hashlib.sha256(json.dumps(_emb, sort_keys=True).encode("utf-8")).hexdigest()[:12]}
else:
    V = np.load(Path(INDEX_DIR) / "vectors.npy")
    chunks = [json.loads(l) for l in open(Path(INDEX_DIR) / "chunks.jsonl", encoding="utf-8")]
    manifest = json.loads((Path(INDEX_DIR) / "manifest.json").read_text(encoding="utf-8"))
assert len(V) == len(chunks) and V.dtype == np.float32, "one float32 vector per chunk row"
# row i of V <-> line i of chunks.jsonl. seq restarts per document in multi-document corpora: never assume seq == i.
assert len({c["chunk_id"] for c in chunks}) == len(chunks), "chunk_ids must be unique"
_last = {}
for c in chunks:
    assert c["seq"] > _last.get(c["doc_id"], -1), "(doc_id, seq) must increase within each document"
    _last[c["doc_id"]] = c["seq"]
assert np.allclose(np.linalg.norm(V, axis=1), 1, atol=1e-5), "rows of V must be unit length"
print(f"index: {len(chunks)} chunks, V {V.shape} {V.dtype}, snapshot {SNAP}, params_hash {chunks[0]['params_hash']}, embed_hash {manifest.get('embed_hash')}")
print("chunk 0 id:", chunks[0]["chunk_id"], "| page", chunks[0]["page_segments"][0]["page"])

## Seeds and judging helpers (given, see 06 §6.7 and P6)

`SEEDS` are the 10 **draft** seed items of 06 §6.7 (`llm-draft`: fine for a lab, never a scored result). Each answerable item has gold spans, converted here to **document offsets**.
- `rel(chunk, item)`: 1 if the chunk overlaps a gold span (06 §7.2 `any_overlap`), else 0.
- `ce(chunk_list, item)`: 1 if the chunks **together** cover every evidence group (Complete Evidence, 06 §7.2).
- `n_tokens(text)`: tokens with `o200k_harmony` (the tokenizer family of the Groq model; P1).

In [ ]:
md06 = (WS / "06_RAG_PROJECT_AND_EVALUATION_PLAN.md").read_text(encoding="utf-8")
SEEDS = [json.loads(l) for l in _blocks(_section(md06, "6.7 Draft seed"), "jsonl")[0].splitlines() if l.strip()]
for s in SEEDS:
    s["G"] = [[[(starts[g["page"] - 1] + g["char_start"], starts[g["page"] - 1] + g["char_end"]) for g in alt["segments"]]
               for alt in grp["alternatives"]] for grp in s["evidence_groups"]]
ANSWERABLE = [s for s in SEEDS if s["answerable"]]
QUESTION = {s["qid"]: s["question"] for s in SEEDS}
Q_D002 = QUESTION["D-002"]

def _covered(a, b, ranges):
    """Fraction of non-whitespace characters of doc_text[a:b] inside the union of `ranges`."""
    idx = [i for i in range(a, b) if not doc_text[i].isspace()]
    return sum(any(x <= i < y for x, y in ranges) for i in idx) / max(1, len(idx))

def rel(chunk, item):
    x, y = chunk["char_start"], chunk["char_end"]
    return int(any(any(not doc_text[i].isspace() for i in range(max(a, x), min(b, y)))
                   for grp in item["G"] for alt in grp for a, b in alt))

def ce(chunk_list, item):
    ranges = [(c["char_start"], c["char_end"]) for c in chunk_list]
    return int(all(any(all(_covered(a, b, ranges) == 1.0 for a, b in alt) for alt in grp) for grp in item["G"]))

import tiktoken
_enc = tiktoken.get_encoding("o200k_harmony")
def n_tokens(text):
    return len(_enc.encode(text))

print(len(SEEDS), "seeds,", len(ANSWERABLE), "answerable:", [s["qid"] for s in SEEDS])

## Task 1: embed the question, then score every chunk (MC-002)

**Predict first.** For the D-002 question, what is the shape of the query vector, and what is the shape of `V @ q`? (The index has `len(chunks)` rows.)

In [ ]:
my_prediction_q_shape = None        # e.g. (12,)  replace BEFORE you run anything
my_prediction_scores_shape = None   # e.g. (12, 12)

Now build, for `Q_D002`:
- `q`: the query vector, from **the same embedder that built the index**, on the **query side** of its interface;
- `scores`: one cosine score per chunk (the rows of `V` and `q` are unit length, so a dot product is the cosine).

In [ ]:
# Your code here (t1). Blank-file first: write it yourself; no copying from c4_measure.py.

In [ ]:
# CHECK Task 1 (do not edit)
assert my_prediction_q_shape is not None and my_prediction_scores_shape is not None, "write both predictions first"
assert isinstance(q, np.ndarray) and q.shape == (V.shape[1],), f"q must be one vector of shape ({V.shape[1]},), got {getattr(q, 'shape', None)}"
assert abs(float(np.linalg.norm(q)) - 1) < 1e-5, "q must be unit length (cosine = dot product needs unit vectors on both sides)"
assert np.allclose(q, embedder.embed_query(Q_D002), atol=1e-6), "q must be embedder.embed_query(Q_D002): the same model, the query side"
assert isinstance(scores, np.ndarray) and scores.shape == (len(chunks),), \
    f"one score per chunk: expected shape ({len(chunks)},), got {getattr(scores, 'shape', None)}. Not a chunk x chunk matrix (MC-002)"
print(f"Task 1 OK. Your predictions {my_prediction_q_shape}, {my_prediction_scores_shape}; measured {q.shape}, {scores.shape}")

## Task 2: `retrieve` that returns scores (the C4 → C5 interface)

Write `retrieve(question, embedder, V, chunks, k=4)`. It returns a **list of dicts, best first**, one per retrieved chunk, with exactly these keys:

| key | value |
|---|---|
| `rank` | 1-based position (1, 2, ...) |
| `chunk_id` | `chunks[i]["chunk_id"]` |
| `doc_id` | `chunks[i]["doc_id"]` |
| `page` | `chunks[i]["page_segments"][0]["page"]` (per-page chunks have one segment) |
| `char_start`, `char_end` | document offsets of the chunk |
| `score` | the cosine score, a plain Python `float` |
| `text` | the chunk text |

Rules: embed the question on the **query side**; exact search over all rows; highest score first; **ties broken by the smaller `chunk_id`** (06 §3.1), not by row position; if `k` exceeds the number of chunks, return them all. You may reuse your P5 `top_k`.

In [ ]:
# Your code here (t2). Blank-file first: write it yourself; no copying from c4_measure.py.

In [ ]:
# CHECK Task 2a (do not edit): shape, order, scores
res = retrieve(Q_D002, embedder, V, chunks, k=K)
KEYS = {"rank", "chunk_id", "doc_id", "page", "char_start", "char_end", "score", "text"}
assert isinstance(res, list) and len(res) == K, f"return a list of {K} dicts"
assert all(isinstance(r, dict) and set(r) == KEYS for r in res), f"each result needs exactly the keys {sorted(KEYS)}"
assert [r["rank"] for r in res] == [1, 2, 3, 4], "ranks are 1-based: 1, 2, 3, 4"
assert all(type(r["score"]) is float for r in res), "score must be a plain Python float (json-serialisable), and it must be there"
_row = {c["chunk_id"]: i for i, c in enumerate(chunks)}
_s = V @ embedder.embed_query(Q_D002)
assert all(abs(r["score"] - float(_s[_row[r["chunk_id"]]])) < 1e-6 for r in res), "each score must be the cosine of that chunk"
assert all(a["score"] >= b["score"] for a, b in zip(res, res[1:])), "scores must be non-increasing: best first"
assert _fp([r["chunk_id"] for r in res]) == "78ebfed94d0021da", "the D-002 top-4 is not right yet (direction? query side?)"
json.dumps(res)                                         # the C5 hand-off must be plain JSON types
assert len(retrieve(Q_D002, embedder, V[:3], chunks[:3], k=10)) == 3, "k larger than the index: return every chunk"
print("Task 2a OK:", [(r["rank"], r["page"], round(r["score"], 4)) for r in res])

In [ ]:
# CHECK Task 2b (do not edit): the query side of the embedder
class _Spy:
    """Behaves like an asymmetric embedder (Cohere-style): the document side gives a DIFFERENT vector."""
    def __init__(self, inner):
        self.inner, self.calls = inner, []
        self.model_id, self.dim = inner.model_id, inner.dim
    def embed_query(self, text):
        self.calls.append("query"); return self.inner.embed_query(text)
    def embed_documents(self, texts):
        self.calls.append("documents")
        r = np.random.default_rng(0).standard_normal((len(texts), self.dim)).astype(np.float32)
        return r / np.linalg.norm(r, axis=1, keepdims=True)
_spy = _Spy(embedder)
_r = retrieve(Q_D002, _spy, V, chunks, k=K)
assert _spy.calls == ["query"], f"retrieve must call embed_query once and never embed_documents; calls were {_spy.calls}"
assert [r["chunk_id"] for r in _r] == [r["chunk_id"] for r in res], "same model, query side: same result"
print("Task 2b OK: the question went through embed_query only")

In [ ]:
# CHECK Task 2c (do not edit): ties go to the smaller chunk_id, whatever the row order
_N = len(chunks)
_cd = chunks + [{**c, "seq": c["seq"] + _N, "chunk_id": c["chunk_id"][:-5] + f"{c['seq'] + _N:05d}"} for c in chunks]
_perm = np.random.default_rng(7).permutation(2 * _N)
_Vd, _cd = np.vstack([V, V])[_perm], [_cd[i] for i in _perm]          # every chunk twice, rows shuffled
_rd = retrieve(Q_D002, embedder, _Vd, _cd, k=K)
_seqs = [int(r["chunk_id"][-5:]) for r in _rd]
assert _rd[0]["score"] == _rd[1]["score"], "the duplicated rows must tie exactly (same vector, same query)"
assert _fp(_seqs) == "65b8cbacc1598a5b", f"tie order {_seqs}: equal scores must go to the smaller chunk_id, not to the earlier row"
print("Task 2c OK: tied copies ordered by chunk_id:", _seqs)

## Task 3: the score table (5+ queries; the 02 C4 evidence)

**Predict first.** For D-002 ("In what year was the Paris Agreement adopted?"), the year is on **p.10**. At which rank will the first chunk that overlaps that sentence appear?

In [ ]:
my_prediction_d002_rank = None   # an integer, BEFORE you build the table

For **every** seed in `SEEDS`, retrieve the top 8 with your `retrieve` and build:
- `table`: a list of rows `{"qid", "rank", "seq", "page", "score", "rel"}` (8 rows per seed; `rel` is `rel(chunk, item)` for answerable items, `None` for the two unanswerable ones);
- `first_rel`: `{qid: rank of the first relevant chunk}` for the 8 answerable seeds (use the full ranking if it is not in the top 8);
- `first_ce`: `{qid: the smallest k at which ce(top_k, item) == 1}` for the 8 answerable seeds.

Then print the table one seed at a time and **look at the scores**: the gap between rank 1 and 2, the gap between rank 4 and 5, and how far the first relevant chunk sits below rank 1.

In [ ]:
# Your code here (t3). Blank-file first: write it yourself; no copying from c4_measure.py.

In [ ]:
# CHECK Task 3 (do not edit)
assert my_prediction_d002_rank is not None, "write your prediction first"
assert isinstance(table, list) and len(table) == 8 * len(SEEDS), "8 rows per seed"
assert all(set(r) >= {"qid", "rank", "seq", "page", "score", "rel"} for r in table), "each row needs qid, rank, seq, page, score, rel"
assert _fp({k: int(v) for k, v in first_rel.items()}) == "a8aa456e76ccea3b", "first_rel is not right yet: check the ranking and rel()"
assert _fp({k: int(v) for k, v in first_ce.items()}) == "179e0af8b77aa72a", "first_ce is not right yet: ce() needs the whole top-k list"
print(f"Task 3 OK. Your D-002 prediction: {my_prediction_d002_rank}; measured: {first_rel['D-002']}")

## Task 4: diagnose every case where a distractor beat the evidence

For each answerable seed with `first_rel > 1`:
1. `outranked`: the sorted list of those qids.
2. `near_dup[qid]`: the cosine **between the rank-1 chunk and the first relevant chunk** (`float(V[i] @ V[j])`). A high value means the two chunks look alike to the model.
3. `diagnosis[qid]`: 2 to 4 sentences of your own: which chunk won and by how much (score margin), **why** (open both texts: same topic, same sentence pattern, a near-duplicate paragraph, a mention of a term from the question?), whether k = 4 still catches the evidence, and whether it is a **retrieval** or a **generation** problem (CR-11).

The tutor grades the diagnoses (05 §4.2 Application rubric). This table plus your diagnoses is the C4 mastery evidence.

In [ ]:
# Your code here (t4). Blank-file first: write it yourself; no copying from c4_measure.py.

In [ ]:
# CHECK Task 4 (do not edit)
assert _fp(sorted(outranked)) == "35f37be3c6c19d49", "outranked: every answerable seed whose first relevant rank is > 1"
assert _fp({k: round(float(v), 2) for k, v in near_dup.items()}) == "34d696449531893c", \
    "near_dup: cosine between the rank-1 chunk vector and the first relevant chunk vector"
assert set(diagnosis) == set(outranked), "one diagnosis per outranked seed"
for _q, _t in diagnosis.items():
    assert len(_t.split()) >= 25, f"{_q}: write at least 2 sentences (25+ words)"
    assert re.search(r"retriev|generat", _t, re.I), f"{_q}: say whether it is a retrieval or a generation problem (CR-11)"
print("Task 4 OK (format). The tutor grades the content.")

## Task 5: can a score threshold spot "no answer here"?

D-009 and D-010 have **no answer** in the document.

**Predict first.** Will both unanswerable questions get a lower rank-1 score than every answerable one?

In [ ]:
my_prediction_threshold_works = None   # True or False, BEFORE you compute

Build `top1 = {qid: rank-1 score}` for all 10 seeds, and `n_below`: how many **answerable** seeds have a rank-1 score **below the higher** of the two unanswerable rank-1 scores.

In [ ]:
# Your code here (t5). Blank-file first: write it yourself; no copying from c4_measure.py.

In [ ]:
# CHECK Task 5 (do not edit)
assert my_prediction_threshold_works is not None, "write your prediction first"
assert _fp({k: round(float(v), 2) for k, v in top1.items()}) == "844dde23504ee1fe", "top1: the rank-1 score of each of the 10 seeds"
assert _fp(int(n_below)) == "ef2d127de37b942b", "n_below is not right yet"
print(f"Task 5 OK. Unanswerable rank-1 scores: D-009 {top1['D-009']:.4f}, D-010 {top1['D-010']:.4f}; "
      f"{n_below} of 8 answerable seeds score lower than D-010.")

## Task 6: the k sweep, both sides of the trade-off (MC-005)

**Toy illustration:** 8 draft items, one small PDF. It shows the *shape* of the trade-off; it is **not** an evaluation and does not choose k for v0 (v0 fixes k = 4; k is tuned later on the frozen eval set).

**Predict first.** At k = 1, 2, 4, 8: how many of the 8 answerable seeds will have complete evidence (CE), and how many tokens will the retrieved chunks cost on average?

In [ ]:
my_prediction_ce = None       # e.g. {1: 5, 2: 6, 4: 7, 8: 8}
my_prediction_tokens = None   # e.g. {1: 200, 2: 400, 4: 800, 8: 1600}

Build `sweep = {k: {"hit": int, "ce": int, "tokens": float}}` for k in (1, 2, 4, 8):
- `hit`: answerable seeds with at least one relevant chunk in the top k (Hit@k, P6);
- `ce`: answerable seeds with `ce(top_k, item) == 1`;
- `tokens`: the **mean over all 10 seeds** of `sum(n_tokens(r["text"]) for r in top_k)` (every query costs tokens, answerable or not).

Then write `my_k_sentence`: one or two sentences that name **both** sides of choosing k, with your numbers.

In [ ]:
# Your code here (t6). Blank-file first: write it yourself; no copying from c4_measure.py.

In [ ]:
# CHECK Task 6 (do not edit)
assert my_prediction_ce is not None and my_prediction_tokens is not None, "write both predictions first"
assert set(sweep) == {1, 2, 4, 8}, "sweep needs k = 1, 2, 4, 8"
assert _fp({str(k): [int(sweep[k]["hit"]), int(sweep[k]["ce"])] for k in (1, 2, 4, 8)}) == "84011ffed49b16a9", "hit/ce counts are not right yet"
assert _fp({str(k): round(float(sweep[k]["tokens"])) for k in (1, 2, 4, 8)}) == "f3407b9eb252ba2f", "mean tokens are not right yet (all 10 seeds)"
assert isinstance(my_k_sentence, str) and re.search(r"token|cost", my_k_sentence, re.I) and re.search(r"evidence|CE|recall|hit", my_k_sentence, re.I), \
    "my_k_sentence must name both sides: the evidence you gain and the tokens you pay"
for k in (1, 2, 4, 8):
    print(f"k={k}: Hit {sweep[k]['hit']}/8  CE {sweep[k]['ce']}/8  tokens {sweep[k]['tokens']:.0f}")
print("Task 6 OK. The tutor grades my_k_sentence.")

## Task 7: the wrong query model fails silently (MC-002, boundary case)

`sentence-transformers/paraphrase-MiniLM-L3-v2` is in your local cache. It also makes **384-dimensional** vectors.

**Predict first.** If you embed the questions with it but search the MiniLM-L6 index `V`, will Python raise an error? And how many of the 8 answerable seeds will still have complete evidence at k = 4?

In [ ]:
my_prediction_error = None    # True or False
my_prediction_wrong_ce = None # an integer 0..8

1. Build `wrong_embedder` (a `MiniLMEmbedder` for that model) and compute `wrong = [hit_at_4, ce_at_4]` over the 8 answerable seeds, searching the **unchanged** `V` and `chunks`.
2. Write `check_compatible(embedder, manifest)`: it raises `ValueError` when the embedder's `model_id` differs from `manifest["embedding"]["model_id"]`, and returns `None` otherwise. This one line is what turns a silent failure into a loud one. (C3's manifest also carries an `embed_hash` over provider, model id, revision, input type, normalisation and preprocessing; a stricter guard can compare that too.)

In [ ]:
# Your code here (t7). Blank-file first: write it yourself; no copying from c4_measure.py.

In [ ]:
# CHECK Task 7 (do not edit)
assert my_prediction_error is not None and my_prediction_wrong_ce is not None, "write both predictions first"
assert _fp([int(x) for x in wrong]) == "28e7165fc1adc5b4", "wrong = [Hit@4, CE@4] with the L3 query model on the L6 index"
assert check_compatible(embedder, manifest) is None, "the right embedder must pass"
try:
    check_compatible(wrong_embedder, manifest)
    raise AssertionError("check_compatible must raise ValueError for the wrong model")
except ValueError:
    pass
print(f"Task 7 OK. Wrong query model: Hit@4 {wrong[0]}/8, CE@4 {wrong[1]}/8, and no error from NumPy.")

## Task 8 (optional, OFF by default): Cohere and its `input_type`

This is the **only** cell that calls an API. It embeds every chunk and the 10 questions with the Cohere model named in `.env` (`COHERE_EMBED_MODEL_ID`, or `COHERE_EMBED_MODEL` as your `.env` names it), so it uses your Cohere quota. Run it only when you decide to, with `RUN_COHERE = True`. It never prints the key.

Cohere's API takes an `input_type`: `search_document` for chunks, `search_query` for questions (06 §2.4). With an asymmetric model, embedding the question as a document is a real mismatch (Task 2b simulated it).

Your task, if you run it: compute `cohere_ce = {"query": CE@4, "as_document": CE@4}` over the 8 answerable seeds, using your own `retrieve` with `cohere_q` and then with `cohere_d`. **The result is UNVERIFIED until you run it**; record it in your notes with the date and model id. Never put it in a results table without a logged run.

In [ ]:
RUN_COHERE = False     # set True only when you decide to spend the API calls

class CohereEmbedder:
    """06 §2.3 Embedder for Cohere. query_type lets you simulate the input_type mistake."""
    def __init__(self, query_type="search_query"):
        import cohere
        from dotenv import load_dotenv
        load_dotenv(WS / ".env")
        assert os.getenv("COHERE_API_KEY"), "COHERE_API_KEY missing in .env"
        self._co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
        self.model_id = os.getenv("COHERE_EMBED_MODEL_ID") or os.getenv("COHERE_EMBED_MODEL")
        self.query_type = query_type
    def _embed(self, texts, input_type):
        out = []
        for i in range(0, len(texts), 64):
            r = self._co.embed(texts=list(texts[i:i + 64]), model=self.model_id, input_type=input_type, embedding_types=["float"])
            out += r.embeddings.float_
        X = np.asarray(out, dtype=np.float32)
        return X / np.linalg.norm(X, axis=1, keepdims=True)
    def embed_documents(self, texts):
        return self._embed(texts, "search_document")
    def embed_query(self, text):
        return self._embed([text], self.query_type)[0]

if RUN_COHERE:
    cohere_q, cohere_d = CohereEmbedder("search_query"), CohereEmbedder("search_document")
    V_cohere = cohere_q.embed_documents([c["text"] for c in chunks])
    print("Cohere index:", V_cohere.shape, "model", cohere_q.model_id)
else:
    print("Task 8 skipped (RUN_COHERE = False). No API call was made.")

In [ ]:
# Your code here, only if RUN_COHERE is True (t8). Blank-file first: write it yourself; no copying from c4_measure.py.

## Task 9: the hand-off to C5, in your words

In `my_interface_note`, write 2 to 4 sentences for the C5 lesson: what `retrieve` returns (the keys, their types, the order), and **why the score must travel with each chunk** even though the prompt does not show it. The tutor grades it (Transfer rubric).

In [ ]:
my_interface_note = None   # your sentences

In [ ]:
# CHECK Task 9 (do not edit)
assert isinstance(my_interface_note, str) and len(my_interface_note.split()) >= 30, "write 2-4 sentences (30+ words)"
assert re.search(r"score", my_interface_note, re.I), "say why the score travels with each chunk"
print("Task 9 OK (format). The tutor grades the content.")